In [1]:
import pprint
import radiate as rd
import numpy as np
import polars as pl  # type: ignore

rd.random.seed(123)

In [2]:
def compute(x: float) -> float:
    return 4.0 * x**3 - 3.0 * x**2 + x


inputs = []
answers = []

x = -1.0
for _ in range(20):
    x += 0.1
    inputs.append([x])
    answers.append([compute(x)])

X = np.array(inputs, dtype=np.float32)  # (N, 1)
Y = np.array(answers, dtype=np.float32)  # (N, 1)

Xb = np.concatenate([X, np.ones((X.shape[0], 1), dtype=np.float32)], axis=1)


def fit(weights: list[np.ndarray]) -> float:
    W1 = weights[0].reshape((8, 2))
    W2 = weights[1].reshape((8, 8))
    W3 = weights[2].reshape((1, 8))

    h1 = Xb @ W1.T
    h1 = np.maximum(0, h1)

    h2 = h1 @ W2
    h2 = np.tanh(h2)

    yhat = h2 @ W3.T

    return float(np.mean((yhat - Y) ** 2, dtype=np.float32))

In [3]:
collector = rd.MetricCollector()

engine = (
    rd.Engine.float(
        shape=[16, 64, 8],
        init_range=(-1.0, 1.0),
        bounds=(-3.0, 3.0),
        use_numpy=True,
        dtype=rd.Float32,
    )
    .fitness(fit)
    .subscribe(collector)
    .minimizing()
    .select(rd.Select.boltzmann(temp=4.0))
    .alter(rd.Cross.blend(0.7, 0.4), rd.Mutate.gaussian(0.3))
    .limit(
        rd.Limit.score(0.01), rd.Limit.generations(500), rd.Limit.convergence(50, 0.001)
    )
)

result = engine.run(log=True)

2026-10-11T09:41:44.744606Z  INFO State Change: PreStart -> Running
2026-10-11T09:41:44.748374Z  INFO Epoch 1    | Score:   0.8202 | Time: 3.28ms
2026-10-11T09:41:44.749360Z  INFO Epoch 2    | Score:   0.5925 | Time: 4.22ms
2026-10-11T09:41:44.750210Z  INFO Epoch 3    | Score:   0.4342 | Time: 5.06ms
2026-10-11T09:41:44.751216Z  INFO Epoch 4    | Score:   0.4342 | Time: 6.03ms
2026-10-11T09:41:44.752101Z  INFO Epoch 5    | Score:   0.3404 | Time: 6.88ms
2026-10-11T09:41:44.752961Z  INFO Epoch 6    | Score:   0.2746 | Time: 7.72ms
2026-10-11T09:41:44.753796Z  INFO Epoch 7    | Score:   0.2746 | Time: 8.54ms
2026-10-11T09:41:44.754651Z  INFO Epoch 8    | Score:   0.1894 | Time: 9.38ms
2026-10-11T09:41:44.755487Z  INFO Epoch 9    | Score:   0.1894 | Time: 10.21ms
2026-10-11T09:41:44.756277Z  INFO Epoch 10   | Score:   0.1894 | Time: 10.99ms
2026-10-11T09:41:44.757203Z  INFO Epoch 11   | Score:   0.1894 | Time: 11.88ms
2026-10-11T09:41:44.758085Z  INFO Epoch 12   | Score:   0.1339 | Time: 

In [4]:
metrics = result.metrics()
df = metrics.to_polars()
df

name,last,sum,mean,stddev,var,skew,min,max,count,time_sum,time_mean,time_stddev,time_min,time_max,time_var,generation,update_count,tags
str,f64,f64,f64,f64,f64,f64,f64,f64,i64,duration[μs],duration[μs],duration[μs],duration[μs],duration[μs],duration[μs],i64,i64,list[str]
"""count.evaluation""",80.0,8196.0,80.352943,2.187485,4.78509,11.351353,80.0,100.0,102,null,null,null,null,null,null,100,1,"[""statistic""]"
"""step.evaluate.time""",0.000792,0.073459,0.000364,0.000376,1.4165e-7,0.0,8.3000e-8,0.002101,202,73459µs,363µs,376µs,0µs,2101µs,0µs,100,2,"[""time"", ""step""]"
"""selector.roulette""",20.0,2020.0,20.0,0.0,0.0,0.0,20.0,20.0,101,null,null,null,null,null,null,100,1,"[""selector"", ""statistic""]"
"""selector.roulette.time""",0.000001,0.00019,0.000002,0.000002,4.2529e-12,0.0,0.000001,0.00002,101,189µs,1µs,2µs,1µs,20µs,0µs,100,1,"[""selector"", ""time""]"
"""selector.boltzmann""",80.0,8080.0,80.0,0.0,0.0,0.0,80.0,80.0,101,null,null,null,null,null,null,100,1,"[""selector"", ""statistic""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""mutator.gaussian.rate""",0.3,30.300001,0.3,0.0,0.0,0.0,0.3,0.3,101,null,null,null,null,null,null,100,1,"[""alterer"", ""mutator"", … ""expr""]"
"""step.metric.time""",0.000013,0.001264,0.000013,0.000004,1.8802e-11,0.0,0.00001,0.00005,101,1263µs,12µs,4µs,10µs,50µs,0µs,100,1,"[""time"", ""step""]"
"""time""",0.00091,0.086127,0.000853,0.000252,6.3475e-8,0.0,0.000768,0.003285,101,86127µs,852µs,251µs,767µs,3284µs,0µs,100,1,"[""time""]"


In [ ]:
print(result.metrics().dashboard())

for metric in metrics.values_by_tag(rd.Tag.DERIVED):
    print(metric)

print()
pprint.pprint(metrics["pct.carryover"].to_dict())

[34 metrics, 3337 updates]
----- Metrics ----- (34 :: 3337) 
  carryover: 0.137  diversity: 0.931  unique_members: 93  unique_scores: 93  improvements: 101  iter_time(mean): 852.75µs

== Distributions ==
Name                       | Type   | Mean       | Min        | Max        | N      | Total        | StdDev     | Skew       | Kurt      
---------------------------------------------------------------------------------------------------------------------------------------
age                        | dist   | 0.610      | 0.000      | 9.000      | 100    | 0.061        | 1.974      | 6.269      | 32.574    
genome.size                | dist   | 88.000     | 88.000     | 88.000     | 100    | 8.800        | 0.000      | 0.000      | 0.000     
scores                     | dist   | 4.097      | 0.100      | 53.399     | 100    | 0.410        | 6.921      | 4.713      | 31.911    


== Statistics ==
Name                       | Type   | Mean       | Min        | Max        | N      | Tot

In [ ]:
result.duration()

datetime.timedelta(microseconds=86127)